In [ ]:
import os, shutil
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install monai nibabel scipy -q

import os, json
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from monai.networks.nets import SegResNet
from monai.losses import DiceLoss as MONAIDiceLoss, FocalLoss
from sklearn.model_selection import KFold
import nibabel as nib
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

PREPROCESSED_ROOT = "/content/drive/MyDrive/MIA/7960856/ISLES-2022_preprocessed"
MASK_ROOT         = "/content/drive/MyDrive/MIA/7960856/ISLES-2022/derivatives"
OUTPUT_DIR        = "/content/drive/MyDrive/MIA/ENSEMBLE_RESULTS"
os.makedirs(OUTPUT_DIR, exist_ok=True)
TARGET_SIZE = (128, 128, 128)
print("✅ Setup done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 67.7 MB/s eta 0:00:00


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


Device : cuda
GPU    : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM   : 102.0 GB
✅ Setup done


In [ ]:
def augment_data(img, mask):
    if np.random.rand() > 0.5:
        flip_axis = np.random.randint(1, 4)
        img  = torch.flip(img,  [flip_axis])
        mask = torch.flip(mask, [flip_axis])
    if np.random.rand() > 0.5:
        img = torch.clamp(img + torch.randn_like(img) * 0.05, 0, 1)
    return img, mask

class ISLESDataset(Dataset):
    def __init__(self, case_ids, augment=False):
        self.case_ids = case_ids
        self.augment  = augment
    def __len__(self): return len(self.case_ids)
    def __getitem__(self, idx):
        cid         = self.case_ids[idx]
        folder_name = f"{cid}_ses-0001_dwi"
        img_path    = os.path.join(PREPROCESSED_ROOT, folder_name, f"{folder_name}_img.nii.gz")
        img = nib.load(img_path).get_fdata()
        img = np.transpose(img, (3,0,1,2)) if img.ndim==4 else np.expand_dims(img, 0)
        img = torch.tensor(img[:3], dtype=torch.float32)
        img = F.interpolate(img.unsqueeze(0), size=TARGET_SIZE,
                            mode="trilinear", align_corners=False).squeeze(0)
        mask_path = os.path.join(MASK_ROOT, cid, "ses-0001", f"{cid}_ses-0001_msk.nii.gz")
        mask = nib.load(mask_path).get_fdata()
        mask = torch.tensor(np.expand_dims(mask, 0), dtype=torch.float32)
        mask = F.interpolate(mask.unsqueeze(0), size=TARGET_SIZE,
                             mode="nearest").squeeze(0)
        if self.augment:
            img, mask = augment_data(img, mask)
        return img, mask, cid

def compute_dice(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).float()
    return 2.0*(pred_bin*target).sum().item() / (pred_bin.sum().item()+target.sum().item()+1e-5)

def compute_f1(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).float()
    tp = (pred_bin * target).sum().item()
    fp = (pred_bin * (1-target)).sum().item()
    fn = ((1-pred_bin) * target).sum().item()
    p  = tp/(tp+fp+1e-5); r = tp/(tp+fn+1e-5)
    return 2*p*r/(p+r+1e-5)

def build_segresnet():
    return SegResNet(spatial_dims=3, in_channels=3, out_channels=1,
                     init_filters=32).to(device)

def run_segresnet_fold(train_cases, val_cases, save_path, epochs=30, patience=8):
    dice_loss  = MONAIDiceLoss(to_onehot_y=False, sigmoid=True)
    focal_loss = FocalLoss(alpha=0.5, gamma=2.0)
    model      = build_segresnet()
    optimizer  = Adam(model.parameters(), lr=1e-4)      # ← stable LR
    scheduler  = CosineAnnealingLR(optimizer, T_max=epochs)
    train_loader = DataLoader(ISLESDataset(train_cases, augment=True),
                              batch_size=1, shuffle=True,   # ← batch=1
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(ISLESDataset(val_cases),
                              batch_size=1, shuffle=False,
                              num_workers=2, pin_memory=True)
    best_dice, patience_counter = 0, 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, masks, _ in tqdm(train_loader, desc=f"Ep {epoch+1}/{epochs}", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = 0.7*dice_loss(out, masks) + 0.3*focal_loss(out, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)  # ← tight clip
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        model.eval()
        val_dice, val_f1 = 0, 0
        with torch.no_grad():
            for imgs, masks, _ in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                preds = model(imgs)
                val_dice += compute_dice(preds, masks)
                val_f1   += compute_f1(preds, masks)
        val_dice /= len(val_loader)
        val_f1   /= len(val_loader)
        print(f"Ep {epoch+1:3d} | Loss {total_loss/len(train_loader):.4f} | Val Dice {val_dice:.4f} | F1 {val_f1:.4f}")
        if val_dice > best_dice:
            best_dice, patience_counter = val_dice, 0
            torch.save(model.state_dict(), save_path)
            print(f"  ✅ Saved (Dice: {best_dice:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  ⏹ Early stop at epoch {epoch+1}"); break
    return best_dice

# splits — same seed as before so folds match exactly
all_cases = np.array(sorted([
    f.replace("_ses-0001_dwi","")
    for f in os.listdir(PREPROCESSED_ROOT)
    if f.endswith("_ses-0001_dwi")
]))
kf     = KFold(n_splits=5, shuffle=True, random_state=42)
splits = list(kf.split(all_cases))
print(f"✅ {len(all_cases)} cases | SegResNet ready")

✅ 250 cases | SegResNet ready


In [ ]:
print("="*50 + "\n  SegResNet — Fold 1\n" + "="*50)
tr_idx, val_idx = splits[0]
seg_f1_dice = run_segresnet_fold(
    train_cases = list(all_cases[tr_idx]),
    val_cases   = list(all_cases[val_idx]),
    save_path   = os.path.join(OUTPUT_DIR, "segresnet_fold1_best.pt")
)
print(f"\n✅ SegResNet Fold 1 Best Dice: {seg_f1_dice:.4f}")

  SegResNet — Fold 1


Ep   1 | Loss 0.6913 | Val Dice 0.1627 | F1 0.1627
  ✅ Saved (Dice: 0.1627)


Ep   2 | Loss 0.6827 | Val Dice 0.1996 | F1 0.1996
  ✅ Saved (Dice: 0.1996)


Ep   3 | Loss 0.6769 | Val Dice 0.1717 | F1 0.1717


Ep   4 | Loss 0.6691 | Val Dice 0.1605 | F1 0.1605


...

Ep  26 | Loss 0.3677 | Val Dice 0.4961 | F1 0.4961


Ep  27 | Loss 0.3663 | Val Dice 0.4979 | F1 0.4979


Ep  28 | Loss 0.3639 | Val Dice 0.5017 | F1 0.5017


Ep  29 | Loss 0.3603 | Val Dice 0.5026 | F1 0.5026


Ep  30 | Loss 0.3592 | Val Dice 0.5040 | F1 0.5040

✅ SegResNet Fold 1 Best Dice: 0.5118


In [ ]:
print("="*50 + "\n  SegResNet — Fold 2\n" + "="*50)
tr_idx, val_idx = splits[1]
seg_f2_dice = run_segresnet_fold(
    train_cases = list(all_cases[tr_idx]),
    val_cases   = list(all_cases[val_idx]),
    save_path   = os.path.join(OUTPUT_DIR, "segresnet_fold2_best.pt")
)
print(f"\n✅ SegResNet Fold 2 Best Dice: {seg_f2_dice:.4f}")

  SegResNet — Fold 2


Ep   1 | Loss 0.6892 | Val Dice 0.2124 | F1 0.2124
  ✅ Saved (Dice: 0.2124)


Ep   2 | Loss 0.6805 | Val Dice 0.3142 | F1 0.3142
  ✅ Saved (Dice: 0.3142)


Ep   3 | Loss 0.6735 | Val Dice 0.2882 | F1 0.2882


Ep   4 | Loss 0.6642 | Val Dice 0.3410 | F1 0.3410
  ✅ Saved (Dice: 0.3410)


...

Ep  26 | Loss 0.3706 | Val Dice 0.5803 | F1 0.5803
  ✅ Saved (Dice: 0.5803)


Ep  27 | Loss 0.3681 | Val Dice 0.5810 | F1 0.5810
  ✅ Saved (Dice: 0.5810)


Ep  28 | Loss 0.3654 | Val Dice 0.5792 | F1 0.5792


Ep  29 | Loss 0.3642 | Val Dice 0.5791 | F1 0.5791


Ep  30 | Loss 0.3604 | Val Dice 0.5804 | F1 0.5804

✅ SegResNet Fold 2 Best Dice: 0.5810


In [ ]:
print("="*50 + "\n  SegResNet — Fold 3\n" + "="*50)
tr_idx, val_idx = splits[2]
seg_f3_dice = run_segresnet_fold(
    train_cases = list(all_cases[tr_idx]),
    val_cases   = list(all_cases[val_idx]),
    save_path   = os.path.join(OUTPUT_DIR, "segresnet_fold3_best.pt")
)
print(f"\n✅ SegResNet Fold 3 Best Dice: {seg_f3_dice:.4f}")
print(f"\n🏁 SegResNet Mean: {np.mean([seg_f1_dice,seg_f2_dice,seg_f3_dice]):.4f}")

  SegResNet — Fold 3


Ep   1 | Loss 0.6947 | Val Dice 0.1975 | F1 0.1975
  ✅ Saved (Dice: 0.1975)


Ep   2 | Loss 0.6891 | Val Dice 0.2258 | F1 0.2258
  ✅ Saved (Dice: 0.2258)


Ep   3 | Loss 0.6834 | Val Dice 0.2585 | F1 0.2585
  ✅ Saved (Dice: 0.2585)


Ep   4 | Loss 0.6762 | Val Dice 0.2296 | F1 0.2296


...

Ep  18 | Loss 0.4445 | Val Dice 0.5475 | F1 0.5475


Ep  19 | Loss 0.4381 | Val Dice 0.5563 | F1 0.5563


Ep  20 | Loss 0.4259 | Val Dice 0.5936 | F1 0.5936
  ✅ Saved (Dice: 0.5936)


Ep  21 | Loss 0.4172 | Val Dice 0.5655 | F1 0.5655


Ep 22/30:  75%|███████▌  | 150/200 [04:22<01:27,  1.74s/it]

In [ ]:
import numpy as np

seg_f1_dice = 0.5118  # ← put your actual Fold 1 best Dice here
seg_f2_dice = 0.5810
seg_f3_dice = 0.5936  # ← put your actual Fold 3 best Dice here

seg_folds = {
    1: seg_f1_dice,
    2: seg_f2_dice,
    3: seg_f3_dice,
}

best_fold = max(seg_folds, key=seg_folds.get)
best_dice = seg_folds[best_fold]
mean_dice = np.mean(list(seg_folds.values()))
std_dice  = np.std(list(seg_folds.values()))

print("="*50)
print("  SegResNet 3-Fold Cross-Validation Summary")
print("="*50)
for f, d in seg_folds.items():
    print(f"  Fold {f}: Best Dice = {d:.4f}")
print("-"*50)
print(f"  Best fold     : Fold {best_fold} (Dice = {best_dice:.4f})")
print(f"  Mean Dice     : {mean_dice:.4f} ± {std_dice:.4f}")
print("="*50)

  SegResNet 3-Fold Cross-Validation Summary
  Fold 1: Best Dice = 0.5118
  Fold 2: Best Dice = 0.5810
  Fold 3: Best Dice = 0.5936
--------------------------------------------------
  Best fold     : Fold 3 (Dice = 0.5936)
  Mean Dice     : 0.5621 ± 0.0360
